In [2]:
import os
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool

In [3]:
DATASET_ROOT = Path("../data/raw/breakhis/BreaKHis_v1/histology_slides/breast")

IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif")

image_paths = []
labels = []

for root, dirs, files in os.walk(DATASET_ROOT):
    for file in files:
        if file.lower().endswith(IMAGE_EXTENSIONS):
            full_path = os.path.join(root, file)
            image_paths.append(full_path)

            # Label extraction from folder name
            if "benign" in root.lower():
                labels.append(0)
            elif "malignant" in root.lower():
                labels.append(1)

# IMPORTANT: sort together to preserve order
image_paths, labels = zip(*sorted(zip(image_paths, labels)))
image_paths = list(image_paths)
labels = list(labels)

print("Total images:", len(image_paths))
print("Benign:", labels.count(0))
print("Malignant:", labels.count(1))

Total images: 7909
Benign: 2480
Malignant: 5429


In [4]:
def build_graph_from_image(img_path, img_size=224, dist_thresh=30):
    
    img = cv2.imread(img_path)
    if img is None:
        return None
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    
    _, thresh = cv2.threshold(
        blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    
    kernel = np.ones((3, 3), np.uint8)
    open_img = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    close_img = cv2.morphologyEx(open_img, cv2.MORPH_CLOSE, kernel, iterations=2)
    
    contours, _ = cv2.findContours(
        close_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    
    centroids = []
    node_features = []
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area <= 0:
            continue
        
        perimeter = cv2.arcLength(cnt, True)
        circularity = 4*np.pi*area/(perimeter**2) if perimeter > 0 else 0
        
        x, y, w, h = cv2.boundingRect(cnt)
        
        aspect_ratio = float(w) / h if h > 0 else 0
        
        rect_area = w * h
        extent = float(area) / rect_area if rect_area > 0 else 0
        
        M = cv2.moments(cnt)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            centroids.append((cx, cy))
            
            node_features.append([
                area,
                perimeter,
                circularity,
                aspect_ratio,
                extent
            ])
    
    if len(node_features) < 2:
        return None
    
    node_features = np.array(node_features, dtype=np.float32)
    
    # Normalize features (important)
    node_features = (node_features - node_features.mean(axis=0)) / (node_features.std(axis=0) + 1e-8)
    
    node_features = torch.tensor(node_features, dtype=torch.float)
    
    edge_index = []
    for i, c1 in enumerate(centroids):
        for j, c2 in enumerate(centroids):
            if i != j:
                dist = np.linalg.norm(np.array(c1) - np.array(c2))
                if dist <= dist_thresh:
                    edge_index.append([i, j])
    
    if len(edge_index) == 0:
        return None
    
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
    return Data(x=node_features, edge_index=edge_index)

In [5]:
class CellularGNN(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 32)
        self.conv2 = GCNConv(32, 64)
        self.classifier = torch.nn.Linear(64, 2)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)
        logits = self.classifier(x)
        return logits, x

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CellularGNN(in_channels=5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

In [7]:
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for img_path, label in tqdm(zip(image_paths, labels), total=len(image_paths)):
        
        data = build_graph_from_image(img_path)
        if data is None:
            continue
        
        data = data.to(device)
        batch = torch.zeros(data.x.size(0), dtype=torch.long).to(device)
        
        optimizer.zero_grad()
        
        logits, embedding = model(data.x, data.edge_index, batch)
        
        loss = criterion(logits, torch.tensor([label], device=device))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

100%|██████████████████████████████████████████████████████████████████████████████| 7909/7909 [02:37<00:00, 50.17it/s]


Epoch 1, Loss: 141.4491


100%|██████████████████████████████████████████████████████████████████████████████| 7909/7909 [02:53<00:00, 45.62it/s]


Epoch 2, Loss: 450.4534


100%|██████████████████████████████████████████████████████████████████████████████| 7909/7909 [02:56<00:00, 44.77it/s]


...

Epoch 18, Loss: 318.2941


100%|██████████████████████████████████████████████████████████████████████████████| 7909/7909 [02:45<00:00, 47.78it/s]


Epoch 19, Loss: 252.6107


100%|██████████████████████████████████████████████████████████████████████████████| 7909/7909 [02:45<00:00, 47.79it/s]

Epoch 20, Loss: 253.7639


In [8]:
os.makedirs("../models", exist_ok=True)

torch.save(model.state_dict(), "../models/cellular_gnn_trained.pt")

print("Trained GNN saved.")

Trained GNN saved.


In [9]:
model.eval()

all_graph_embeddings = []

for img_path in tqdm(image_paths):
    
    data = build_graph_from_image(img_path)
    
    if data is None:
        all_graph_embeddings.append(np.zeros(64))
        continue
    
    data = data.to(device)
    batch = torch.zeros(data.x.size(0), dtype=torch.long).to(device)
    
    with torch.no_grad():
        logits, embedding = model(data.x, data.edge_index, batch)
    
    all_graph_embeddings.append(embedding.cpu().numpy().squeeze())

all_graph_embeddings = np.array(all_graph_embeddings)

print("Final graph embedding shape:", all_graph_embeddings.shape)

100%|██████████████████████████████████████████████████████████████████████████████| 7909/7909 [02:36<00:00, 50.54it/s]

Final graph embedding shape: (7909, 64)


In [10]:
os.makedirs("../outputs/gnn_features", exist_ok=True)

np.save("../outputs/gnn_features/tumor_graph_embeddings.npy",
        all_graph_embeddings)

print("Saved embeddings.")

Saved embeddings.
